# 🐍 A Snake agent that scales

This agent was trained **only on boards 6×6 through 22×22**, yet it plays boards it has *never seen* — up to **100×100** — zero-shot. It's tiny: ~209K parameters, 824 KB.

**Runtime → Run all**, then set `BOARD_SIZE` in the last cell and watch it play.

Full story: [github.com/Saheb/rl-snake](https://github.com/Saheb/rl-snake) · [the investigation](https://github.com/Saheb/rl-snake/blob/main/size_transfer/FINDINGS.md).

In [ ]:
# Download the shipped agent (824 KB) and set up.
import os, urllib.request
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

MODEL_URL = "https://raw.githubusercontent.com/Saheb/rl-snake/main/size_transfer/curriculum_ego_best.pth"
MODEL_PATH = "curriculum_ego_best.pth"
if not os.path.exists(MODEL_PATH):
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print("model ready:", os.path.getsize(MODEL_PATH), "bytes")

In [ ]:
# Minimal Snake environment (faithful to the training env: 0=up 1=right 2=down 3=left).
import random as rnd, collections

class SnakeGame:
    def __init__(self, board_size=10):
        self.board_size = board_size
        self.reset()

    def _pick_food(self):
        avail = set((x, y) for x in range(self.board_size) for y in range(self.board_size)) - set(self.snake_position)
        return rnd.choice(list(avail)) if avail else None

    def reset(self):
        b = self.board_size
        self.snake_position = collections.deque([(b // 2, b // 2)])
        self.food_position = self._pick_food()
        self.score = 0

    def _move(self, a):
        x, y = self.snake_position[-1]
        return [(x - 1, y), (x, y + 1), (x + 1, y), (x, y - 1)][a]

    def _collision(self, nh, eat=False):
        x, y = nh
        if x < 0 or x >= self.board_size or y < 0 or y >= self.board_size:
            return True
        body = list(self.snake_position)
        if not eat:
            body = body[1:]            # tail moves away unless we just ate
        return nh in body

    def step(self, a):
        """Returns done (bool)."""
        nh = self._move(a)
        eat = (nh == self.food_position)
        if self._collision(nh, eat):
            return True
        if eat:
            self.score += 1
            self.snake_position.append(nh)
            self.food_position = self._pick_food()
            return self.food_position is None
        self.snake_position.append(nh)
        self.snake_position.popleft()
        return False

In [ ]:
# The agent: egocentric input + attention pool + dueling head. Runs on any board size.
class CrossSizeAgent(nn.Module):
    def __init__(self, n_actions=4, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.conv = nn.Sequential(
            nn.Conv2d(5, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
        )
        self.attn = nn.Conv2d(128, n_heads, 1)
        self.proj = nn.Sequential(nn.Linear(128 * n_heads, 128), nn.ReLU())
        self.value_stream = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1))
        self.advantage_stream = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, n_actions))

    def _pool(self, x):
        h = self.conv(x)
        b, c, hh, ww = h.shape
        w = torch.softmax(self.attn(h).view(b, self.n_heads, hh * ww), dim=2)
        pooled = torch.einsum("bnk,bck->bnc", w, h.view(b, c, hh * ww))
        av = self.proj(pooled.reshape(b, self.n_heads * c))
        return torch.cat([av, h.amax(dim=(2, 3))], dim=1)

    def forward(self, x):
        z = self._pool(x)
        return self.value_stream(z) + (self.advantage_stream(z) - self.advantage_stream(z).mean(1, keepdim=True))

KAPPA = 3.0
def state_ego(game):
    b = game.board_size
    s = np.zeros((3, b, b), np.float32)
    snake = list(game.snake_position)
    for p in snake[:-1]:
        s[0, p[0], p[1]] = 1.0
    head = snake[-1]
    s[1, head[0], head[1]] = 1.0
    if game.food_position:
        s[2, game.food_position[0], game.food_position[1]] = 1.0
    rel_r = np.tanh((np.arange(b) - head[0]) / KAPPA)[:, None] * np.ones((1, b), np.float32)
    rel_c = np.tanh((np.arange(b) - head[1]) / KAPPA)[None, :] * np.ones((b, 1), np.float32)
    return np.concatenate([s, rel_r[None].astype(np.float32), rel_c[None].astype(np.float32)], axis=0)

model = CrossSizeAgent()
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()
print("agent loaded:", sum(p.numel() for p in model.parameters()), "params")

In [ ]:
BOARD_SIZE = 40  # @param {type:"slider", min:6, max:100, step:1}

rnd.seed(0)
game = SnakeGame(BOARD_SIZE)
frames = []

def grid():
    g = np.zeros((BOARD_SIZE, BOARD_SIZE), int)
    for p in list(game.snake_position)[:-1]:
        g[p[0], p[1]] = 1
    h = game.snake_position[-1]
    g[h[0], h[1]] = 2
    if game.food_position:
        g[game.food_position[0], game.food_position[1]] = 3
    return g

with torch.no_grad():
    for _ in range(min(BOARD_SIZE * BOARD_SIZE * 2, 1200)):
        frames.append(grid())
        if game.food_position is None:
            break
        a = int(torch.argmax(model(torch.tensor(state_ego(game)[None]))).item())
        if game.step(a):
            break
frames.append(grid())
print(f"{BOARD_SIZE}x{BOARD_SIZE}: ate {game.score} food over {len(frames)} steps (trained only up to 22x22)")

# Animate (subsample so the inline player stays light).
palette = np.array([[1, 1, 1], [0.20, 0.70, 0.35], [0.05, 0.32, 0.15], [0.90, 0.22, 0.22]])
stride = max(1, len(frames) // 200)
clip = frames[::stride]
fig, ax = plt.subplots(figsize=(6, 6)); ax.axis("off")
im = ax.imshow(palette[clip[0]], interpolation="nearest")
ax.set_title(f"{BOARD_SIZE}x{BOARD_SIZE} - zero-shot")
def _upd(i):
    im.set_data(palette[clip[i]]); return [im]
anim = animation.FuncAnimation(fig, _upd, frames=len(clip), interval=80, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())